# CardioIA – Fase 2, Parte 2: Classificador de Risco (baixo risco / alto risco)

Pipeline: TF-IDF → treino de `LogisticRegression` e `DecisionTreeClassifier` → comparação de métricas → escolha do melhor modelo → teste com frases novas.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option("display.max_colwidth", None)

In [2]:
df = pd.read_csv("../assets/base_risco.csv")

print("Total de frases:", len(df))
print(df["situacao"].value_counts())
df.head()

Total de frases: 60
situacao
alto risco     30
baixo risco    30
Name: count, dtype: int64


,frase,situacao
0,sinto dor no peito e falta de ar,alto risco
1,tive uma dor forte no peito que irradia para o braço esquerdo,alto risco
2,estou com suor frio e aperto no peito há dez minutos,alto risco
3,sinto o coração disparado e quase desmaiei agora há pouco,alto risco
4,tenho falta de ar mesmo em repouso sem fazer nenhum esforço,alto risco


In [3]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    df["frase"], df["situacao"],
    test_size=0.2, stratify=df["situacao"], random_state=42,
)

print("Treino:", len(X_treino), "| Teste:", len(X_teste))
print(y_teste.value_counts())

Treino: 48 | Teste: 12
situacao
alto risco     6
baixo risco    6
Name: count, dtype: int64


In [4]:
vetorizador = TfidfVectorizer(lowercase=True)
X_treino_tfidf = vetorizador.fit_transform(X_treino)
X_teste_tfidf = vetorizador.transform(X_teste)

print("Vocabulário:", len(vetorizador.vocabulary_), "termos")
print("Shape treino:", X_treino_tfidf.shape)

Vocabulário: 210 termos
Shape treino: (48, 210)


In [5]:
modelo_lr = LogisticRegression(random_state=42)
modelo_lr.fit(X_treino_tfidf, y_treino)

modelo_dt = DecisionTreeClassifier(random_state=42)
modelo_dt.fit(X_treino_tfidf, y_treino)

print("Modelos treinados: Logistic Regression e Decision Tree")

Modelos treinados: Logistic Regression e Decision Tree


In [6]:
pred_lr = modelo_lr.predict(X_teste_tfidf)
pred_dt = modelo_dt.predict(X_teste_tfidf)

acc_lr = accuracy_score(y_teste, pred_lr)
acc_dt = accuracy_score(y_teste, pred_dt)

print(f"Acurácia Logistic Regression: {acc_lr:.2f}")
print(f"Acurácia Decision Tree:       {acc_dt:.2f}")

print("\n=== Logistic Regression ===")
print(classification_report(y_teste, pred_lr))
print("Matriz de confusão:\n", confusion_matrix(y_teste, pred_lr, labels=["alto risco", "baixo risco"]))

print("\n=== Decision Tree ===")
print(classification_report(y_teste, pred_dt))
print("Matriz de confusão:\n", confusion_matrix(y_teste, pred_dt, labels=["alto risco", "baixo risco"]))

Acurácia Logistic Regression: 0.75
Acurácia Decision Tree:       0.92

=== Logistic Regression ===
              precision    recall  f1-score   support

  alto risco       0.80      0.67      0.73         6
 baixo risco       0.71      0.83      0.77         6

    accuracy                           0.75        12
   macro avg       0.76      0.75      0.75        12
weighted avg       0.76      0.75      0.75        12

Matriz de confusão:
 [[4 2]
 [1 5]]

=== Decision Tree ===
              precision    recall  f1-score   support

  alto risco       0.86      1.00      0.92         6
 baixo risco       1.00      0.83      0.91         6

    accuracy                           0.92        12
   macro avg       0.93      0.92      0.92        12
weighted avg       0.93      0.92      0.92        12

Matriz de confusão:
 [[6 0]
 [1 5]]


In [7]:
# Decision Tree teve maior acurácia neste split (48 treino / 12 teste) e foi o modelo escolhido.
modelo_final = modelo_dt

frases_novas = [
    "sinto uma dor no peito muito forte que não passa e estou suando frio",
    "tive uma leve dor de cabeça hoje, mas já melhorou depois de descansar",
    "acordei com falta de ar e o coração muito acelerado",
    "senti uma pontada rápida nas costas ao me abaixar para pegar algo",
]

X_novas_tfidf = vetorizador.transform(frases_novas)
predicoes = modelo_final.predict(X_novas_tfidf)

for frase, predicao in zip(frases_novas, predicoes):
    print(f"[{predicao}] {frase}")

[alto risco] sinto uma dor no peito muito forte que não passa e estou suando frio
[baixo risco] tive uma leve dor de cabeça hoje, mas já melhorou depois de descansar
[alto risco] acordei com falta de ar e o coração muito acelerado
[alto risco] senti uma pontada rápida nas costas ao me abaixar para pegar algo


## Reflexão sobre padrões e distorções

- **Decision Tree (92%) superou Logistic Regression (75%)** neste split, mas com apenas 12 frases de teste o resultado é sensível ao acaso — não deve ser generalizado como "Decision Tree é sempre melhor" para este problema.
- No teste com frases novas, o modelo classificou corretamente 3 de 4 frases, mas **errou** "senti uma pontada rápida nas costas ao me abaixar para pegar algo" como *alto risco*, apesar de ser um sintoma banal sem nenhuma palavra cardíaca.
- **Causa provável:** a base de treino tem poucas frases de dor osteomuscular nas costas rotuladas como baixo risco, então o modelo associou palavras como "pontada" e "costas" (raras no treino) a um padrão mais próximo da classe alto risco.
- **Limitação de fundo:** 60 frases é uma base muito pequena para um TF-IDF generalizar bem a vocabulário fora do que foi visto — em um cenário real de triagem clínica, esse tipo de falso positivo/negativo teria consequências sérias, reforçando a necessidade de bases muito maiores, validação clínica e supervisão humana antes de qualquer uso real.